In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from datasets import load_dataset
from PIL import Image, ImageChops
import numpy as np

In [ ]:
# --- Configuration & Setup ---
HF_DATASET = 'TheKernel01/AIGC-Detection-Benchmark'
IN_COLAB = 'google.colab' in sys.modules
CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv

    load_dotenv()
    CACHE_DIR = None
    HF_TOKEN = os.getenv('HF_TOKEN')

BATCH_SIZE = 128
IMAGE_SIZE = (224, 224)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# --- ELA Logic ---
def get_ela_image(image, quality=90):
    """Calculates the Error Level Analysis of a PIL Image."""
    original = image.convert('RGB').resize(IMAGE_SIZE)

    # Create temporary compressed version in memory
    from io import BytesIO

    buffer = BytesIO()
    original.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    compressed = Image.open(buffer)

    # Calculate difference and scale for visibility
    ela_img = ImageChops.difference(original, compressed)
    extrema = ela_img.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff

    # Enhance the difference
    return Image.fromarray(np.uint8(np.array(ela_img) * scale))

In [4]:
class ELADataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Convert to ELA before applying tensor transforms
        ela_img = get_ela_image(item['image'])
        if self.transform:
            ela_img = self.transform(ela_img)
        label = torch.tensor(item['label'], dtype=torch.long)
        return ela_img, label

In [5]:
# --- Data Loading ---
print(f'Loading dataset: {HF_DATASET}...')
train_data = load_dataset(
    HF_DATASET, token=HF_TOKEN, cache_dir=CACHE_DIR, split='train'
)
val_data = load_dataset(
    HF_DATASET, token=HF_TOKEN, cache_dir=CACHE_DIR, split='validation'
)

transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]
)

train_loader = DataLoader(
    ELADataset(train_data, transform), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(ELADataset(val_data, transform), batch_size=BATCH_SIZE)

Loading dataset: TheKernel01/AIGC-Detection-Benchmark...


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

In [6]:
# --- Model Definition ---
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)  # Binary: Real (0) vs Fake (1)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [7]:
# --- Training Loop ---
epochs = 5

print(f'Starting training on {DEVICE}...')
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 20 == 0:
            print(
                f'Epoch [{epoch + 1}/{epochs}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}'
            )

    # Save Checkpoint
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'ela_model_epoch_{epoch + 1}.pth')
    torch.save(
        {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': running_loss / len(train_loader),
        },
        checkpoint_path,
    )
    print(f'Checkpoint saved: {checkpoint_path}')


Starting training on cuda...
Epoch [1/5], Step [0/587], Loss: 0.7162
Epoch [1/5], Step [20/587], Loss: 0.5258
Epoch [1/5], Step [40/587], Loss: 0.4603
Epoch [1/5], Step [60/587], Loss: 0.4778
Epoch [1/5], Step [80/587], Loss: 0.3831
Epoch [1/5], Step [100/587], Loss: 0.4372
Epoch [1/5], Step [120/587], Loss: 0.3460
Epoch [1/5], Step [140/587], Loss: 0.2883
Epoch [1/5], Step [160/587], Loss: 0.3565
Epoch [1/5], Step [180/587], Loss: 0.2768
Epoch [1/5], Step [200/587], Loss: 0.3257
Epoch [1/5], Step [220/587], Loss: 0.2652
Epoch [1/5], Step [240/587], Loss: 0.2732
Epoch [1/5], Step [260/587], Loss: 0.2617
Epoch [1/5], Step [280/587], Loss: 0.2758
Epoch [1/5], Step [300/587], Loss: 0.2832
Epoch [1/5], Step [320/587], Loss: 0.2915
Epoch [1/5], Step [340/587], Loss: 0.3260
Epoch [1/5], Step [360/587], Loss: 0.2985
Epoch [1/5], Step [380/587], Loss: 0.2435
Epoch [1/5], Step [400/587], Loss: 0.1972
Epoch [1/5], Step [420/587], Loss: 0.2280
Epoch [1/5], Step [440/587], Loss: 0.3010
Epoch [1/5]